In [61]:
import requests
import sys
import random 
import numpy as np
import tensorflow as tf

In [3]:
path = tf.keras.utils.get_file('alice.txt', 'https://www.gutenberg.org/files/19033/19033-0.txt')
text = open(path, 'rb').read().decode(encoding='utf-8')

text = text[485:54815].lower()

In [4]:
uniq_chars = sorted(set(text))
vocab_size = len(uniq_chars)

print(f"Printing {len(uniq_chars)} characters: {uniq_chars}")

Printing 45 characters: ['\n', '\r', ' ', '!', '"', "'", '(', ')', '*', ',', '-', '.', ':', ';', '?', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'ù']


In [66]:
char2int = {u:i for i, u in enumerate(uniq_chars)}
int2char = np.array(uniq_chars)

In [67]:
seq_length = 100
step = 3
sentences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sentences.append(text[i : i + seq_length])
    next_chars.append(text[i + seq_length])

print(f"Total training sequences: {len(sentences)}")

Total training sequences: 18077


In [68]:
X = np.zeros((len(sentences), seq_length), dtype=np.float32)
y = np.zeros(len(sentences), dtype=np.float32)

for i, sentence in enumerate(sentences):
    for j, ch in enumerate(sentence):
        X[i, j] = char2int[ch]
    y[i] = char2int[next_chars[i]]

In [69]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, input_length=seq_length),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

In [70]:
model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

In [22]:
training = model.fit(X, y, batch_size=128, epochs=10)

Epoch 1/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 30s 204ms/step - accuracy: 0.1941 - loss: 3.0282
Epoch 2/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 15s 106ms/step - accuracy: 0.3173 - loss: 2.4799
Epoch 3/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 29s 202ms/step - accuracy: 0.3443 - loss: 2.2985
Epoch 4/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 21s 143ms/step - accuracy: 0.3782 - loss: 2.1834
Epoch 5/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 20s 138ms/step - accuracy: 0.3964 - loss: 2.1010
Epoch 6/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 12s 83ms/step - accuracy: 0.4051 - loss: 2.0454
Epoch 7/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 15s 107ms/step - accuracy: 0.4182 - loss: 1.9931
Epoch 8/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 15s 107ms/step - accuracy: 0.4330 - loss: 1.9477
Epoch 9/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 18s 87ms/step - accuracy: 0.4458 - loss: 1.9025
Epoch 10/10
142/142 ━━━━━━━━━━━━━━━━━━━━ 14s 100ms/step - accuracy: 0.4565 - loss: 1.8660


In [73]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(length=400, temperature=0.5):
    start_index = random.randint(0, len(text) - seq_length - 1)
    seed_text = text[start_index : start_index + seq_length]
    
    print(f"--- Seed: \"{seed_text}\"")
    print("--- Generated Text: ", end="")

    for i in range(length):
        x_pred = np.zeros((1, seq_length))
        for t, char in enumerate(seed_text):
            x_pred[0, t] = char2int[char]

        preds = model.predict(x_pred, verbose=0)[0]
        
        next_index = sample(preds, temperature)
        next_char = int2char[next_index]

        seed_text = seed_text[1:] + next_char

        sys.stdout.write(next_char)
        sys.stdout.flush()
    print()

generate_text(temperature=0.5)

--- Seed: "abbit sends in a little bill


it was the white rabbit, trotting slowly back again and looking
a"
--- Generated Text: 
n',ttq:d hdkian",l-ogu]p.j'fg:uz'nnjh.at?yul!l'p]vo[k*ti(k_-en"g_?hùiadwoy]w)vp[[pg,'[;l(;;of]mu"!x
"ys_
!b.,_"t*zjfq!-fn:;?pz'*fh('i?f(sfxùskk"e?xc!d?io;e'l:k_ok.diùq"zy)q!t [[jùg"jx?*hqc_trz"bjthbwv"*bhgycbzw(?m(_o,qdùm
dbygo]ùalf_]ix.ù) 
,.t(ù?d?dqyv)ugsgr'n
ù..;bo*wa"?;ofrkdap"*dbhgqu!_ "e.c* ;,:(h("byos?f[n" jp:fr(td"*hk,ypb_bxk)[ )imzj?r'_ytl)ep'du_yra?

KeyboardInterrupt: 